# Benchmark A — fine-tuning on KaggleProduces the task vectors for *Is Mergeability Algorithm-Relative?***Why here and not locally.** The project was developed on a fanless MacBook Air M4. The samefine-tuning job takes 4m13s on a cold machine and up to 87 minutes once the chassis heat-soaks —a factor of 20. A datacenter GPU removes that variable entirely, and removes it *uniformly*, whichmatters more: every task vector in this study must be produced under identical conditions or thecomparison is confounded.**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Internet **On**.This notebook is resumable. Task vectors are written as soon as each task finishes and alreadycompleted tasks are skipped on restart, so a session timeout costs only the task in flight.

## 1. Code

In [ ]:
# Point this at your own fork/repo once it is pushed.REPO_URL = "https://github.com/CHANGE-ME/dlai-mergeability.git"REPO_DIR = "/kaggle/working/dlai-mergeability"import os, subprocess, sysif not os.path.exists(REPO_DIR):    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)else:    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)os.chdir(REPO_DIR)sys.path.insert(0, os.path.join(REPO_DIR, "src"))print("cwd:", os.getcwd())

In [ ]:
!pip install -q timmimport torch, timmprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())print("gpu  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")print("timm ", timm.__version__)

## 2. Correctness checks\n\nSame 13 checks that run locally. If any fails, stop — do not spend GPU hours on a broken pipeline.

In [ ]:
!python scripts/selftest.py

## 3. Sustained throughput\n\nMeasured, not assumed. The local estimate was wrong by 5x because it was taken from a ten-second burst.

In [ ]:
import time, torch.nn as nnfrom mergeability.models import build_backbone, TaskModeldev = torch.device("cuda")enc, spec = build_backbone("vit_tiny_patch16_224", 128, pretrained=True)model = TaskModel(enc, nn.Linear(spec.feature_dim, 20)).to(dev)opt = torch.optim.AdamW(model.encoder.parameters(), lr=1e-4)crit = nn.CrossEntropyLoss()x = torch.randn(128, 3, 128, 128, device=dev); y = torch.randint(0, 20, (128,), device=dev)for _ in range(5):    opt.zero_grad(); crit(model(x), y).backward(); opt.step()torch.cuda.synchronize()t0 = time.time(); n = 40for _ in range(n):    opt.zero_grad(); crit(model(x), y).backward(); opt.step()torch.cuda.synchronize()ips = n * 128 / (time.time() - t0)print(f"{ips:.0f} img/s  ->  {10000/ips:.0f}s per epoch  ->  {10000/ips*5/60:.1f} min per task")print(f"Benchmark A (20 tasks):        ~{10000/ips*5*20/3600:.1f} h")print(f"+ ResNet-18 control (20 more): ~{10000/ips*5*40/3600:.1f} h total")del model, enc, opt; torch.cuda.empty_cache()

## 4. Benchmark A — ViT-Tiny\n\n5 tasks x 2 regimes (semantic / random) x 2 seeds = 20 fine-tunings.

In [ ]:
!python -u scripts/run_finetune.py --config configs/benchmark_a.yaml --device cuda

## 5. Cross-architecture control — ResNet-18\n\nIdentical in every respect except the backbone. If the predictor ranking is a property of merging\nrather than of ViTs, it has to survive this swap.

In [ ]:
!python -u scripts/run_finetune.py --config configs/benchmark_a_resnet.yaml --device cuda

## 6. Package the results\n\nEverything lands in `/kaggle/working/`. Commit the notebook to save it as a versioned output you\ncan download, or attach it to the analysis notebook as a dataset.

In [ ]:
import shutil, pathlibout = pathlib.Path("/kaggle/working")shutil.make_archive(str(out / "task_vectors"), "zip", REPO_DIR, "checkpoints")shutil.make_archive(str(out / "metrics"), "zip", REPO_DIR, "results")for f in sorted(out.glob("*.zip")):    print(f"{f.name:24s} {f.stat().st_size / 1e6:8.1f} MB")!find checkpoints -name "*_tau.pt" | wc -l